# 17 Python intermediate - `pathlib` v2.0

_Kamil Bartocha_

## Rozkład jazdy

1. 🔹 **`Path` - tworzenie i nawigacja** - konstruktor, operatory, atrybuty
2. 🔹 **Odczyt, zapis i operacje na plikach** - `read_text`, `write_text`, `rename`, `unlink`
3. 🔹 **Przeszukiwanie katalogów** - `iterdir`, `glob`, `rglob`

🐍 Każda sekcja zawiera ćwiczenia.

---

## 1. 🔹 `Path` - tworzenie i nawigacja

`pathlib.Path` reprezentuje scieżkę w systemie plików jako obiekt.
Zastępuje stare podejście z `os.path` i łańcuchami znaków -
kod jest czytelniejszy i mniej podatny na błędy.

| Stare podejście | `pathlib` |
|---|---|
| `os.path.join(a, b)` | `Path(a) / b` |
| `os.path.basename(p)` | `Path(p).name` |
| `os.path.dirname(p)` | `Path(p).parent` |
| `os.path.splitext(p)` | `Path(p).stem`, `.suffix` |
| `os.path.exists(p)` | `Path(p).exists()` |
| `os.makedirs(p)` | `Path(p).mkdir(parents=True)` |

In [ ]:
from pathlib import Path

# Tworzenie obiektow Path
p = Path('/usr/local/bin/python3')
print(p.name)       # python3
print(p.stem)       # python3
print(p.suffix)     # '' (brak rozszerzenia)
print(p.parent)     # /usr/local/bin
print(p.parts)      # ('/', 'usr', 'local', 'bin', 'python3')

# Operator / do laczenia sciezek
base = Path('/home/user')
config = base / 'projects' / 'app' / 'config.yaml'
print(config)   # /home/user/projects/app/config.yaml
print(config.suffix)   # .yaml

In [ ]:
from pathlib import Path

# Sciezka wzgledna i absolutna
rel = Path('data/file.txt')
print(rel.is_absolute())   # False
print(rel.resolve())       # absolutna sciezka (od CWD)

# Sprawdzanie stanu
p = Path('.')
print(p.exists())     # True
print(p.is_dir())     # True
print(p.is_file())    # False

# Biezacy katalog i katalog domowy
print(Path.cwd())    # katalog roboczy
print(Path.home())   # katalog domowy uzytkownika

---

### 🐍 Ćwiczenia - Path i nawigacja

**Ćw. 1.** Dla sciezki `path = Path('/home/user/documents/report.final.pdf')`
wypisz: `name`, `stem`, `suffix`, `parent`, `parent.parent`.

**Ćw. 2.** Zbuduj sciezke do pliku `config.json` w katalogu
`settings/dev/` względem biezacego katalogu. Sprawdź czy istnieje
i wypisz jej absolutną postać.

**Ćw. 3. *(Trudniejsze)*** Napisz funkcję `common_parent(paths)`
zwracającą najdłuższego wspólnego rodzica dla listy sciezek
(np. dla `['/a/b/c', '/a/b/d', '/a/e']` zwraca `Path('/a')`).

In [ ]:
# Ćw. 1
from pathlib import Path

path = Path('/home/user/documents/report.final.pdf')

print(path.name)          # report.final.pdf
print(path.stem)          # report.final
print(path.suffix)        # .pdf
print(path.parent)        # /home/user/documents
print(path.parent.parent) # /home/user

In [ ]:
# Ćw. 2
from pathlib import Path

config = ...
print(config)
print(config.exists())
print(config.resolve())

In [ ]:
# Ćw. 3
from pathlib import Path

def common_parent(paths: list[str]) -> Path:
    # hint: uzyj Path.parts i szukaj wspolnego prefiksu
    ...


paths = ['/home/user/docs/report.pdf', '/home/user/docs/notes.txt', '/home/user/images/photo.jpg']
print(common_parent(paths))   # /home/user

---

## 2. 🔹 Odczyt, zapis i operacje na plikach

`pathlib` ujednolica operacje I/O - nie potrzebujemy `open()` dla
prostych przypadków.

| Metoda | Opis |
|---|---|
| `p.read_text(encoding='utf-8')` | odczyt całego pliku jako str |
| `p.write_text(text, encoding='utf-8')` | zapis str do pliku |
| `p.read_bytes()` | odczyt binarny |
| `p.write_bytes(data)` | zapis binarny |
| `p.rename(target)` | zmiana nazwy / przeniesienie |
| `p.replace(target)` | jak rename, nadpisuje cel |
| `p.unlink()` | usuniecie pliku |
| `p.mkdir(parents=True, exist_ok=True)` | utworzenie katalogu |
| `p.rmdir()` | usuniecie pustego katalogu |
| `shutil.copy(src, dst)` | kopiowanie pliku |

In [ ]:
from pathlib import Path

# Zapis i odczyt tekstu
p = Path('example_pathlib.txt')
p.write_text('Hello, pathlib!\nSecond line.', encoding='utf-8')
content = p.read_text(encoding='utf-8')
print(content)

# Zmiana nazwy
new_p = p.with_name('renamed.txt')
p.rename(new_p)
print(new_p.read_text(encoding='utf-8'))

# Sprzatanie
new_p.unlink()
print(new_p.exists())   # False

In [ ]:
from pathlib import Path

# Tworzenie struktury katalogow
base = Path('temp_project')
(base / 'src').mkdir(parents=True, exist_ok=True)
(base / 'tests').mkdir(parents=True, exist_ok=True)
(base / 'src' / 'main.py').write_text('# main', encoding='utf-8')
(base / 'tests' / 'test_main.py').write_text('# tests', encoding='utf-8')
print(list(base.rglob('*.py')))

# Sprzatanie
import shutil
shutil.rmtree(base)

---

### 🐍 Ćwiczenia - odczyt, zapis, operacje

**Ćw. 4.** Napisz funkcję `count_lines(path)` używającą `Path.read_text()`
zwracającą liczbę niepustych linii w pliku tekstowym.

**Ćw. 5.** Napisz funkcję `backup(path)` tworzącą kopię zapasową pliku:
nazwa kopii to `filename.bak.ext` (np. `data.bak.json`).
Uzyj `path.with_suffix()` i `path.read_bytes()` / `write_bytes()`.

**Ćw. 6. *(Trudniejsze)*** Napisz funkcję `safe_write(path, text)`,
która zapisuje tekst atomowo: najpierw do pliku tymczasowego
`path.with_suffix('.tmp')`, a potem używa `replace()` do zamiany.
Jeśli zapis się nie powiedzie, plik tymczasowy jest usuwany.

In [ ]:
# Ćw. 4
from pathlib import Path

def count_lines(path: Path) -> int:
    ...


# test
p = Path('test_lines.txt')
p.write_text('line1\n\nline3\nline4\n', encoding='utf-8')
print(count_lines(p))   # 3
p.unlink()

In [ ]:
# Ćw. 5
from pathlib import Path

def backup(path: Path) -> Path:
    # hint: with_suffix('.bak' + path.suffix)
    ...


p = Path('data.json')
p.write_bytes(b'{"key": 1}')
bak = backup(p)
print(bak.name)               # data.bak.json
print(bak.read_bytes())       # b'{"key": 1}'
p.unlink()
bak.unlink()

In [ ]:
# Ćw. 6
from pathlib import Path

def safe_write(path: Path, text: str) -> None:
    tmp = path.with_suffix('.tmp')
    try:
        ...
    except Exception:
        if tmp.exists():
            tmp.unlink()
        raise


p = Path('output.txt')
safe_write(p, 'Hello!')
print(p.read_text(encoding='utf-8'))   # Hello!
print(p.with_suffix('.tmp').exists())  # False
p.unlink()

---

## 3. 🔹 Przeszukiwanie katalogów

| Metoda | Opis |
|---|---|
| `p.iterdir()` | bezposrednie dzieci katalogu |
| `p.glob('*.py')` | wzorzec w jednym poziomie |
| `p.rglob('*.py')` | wzorzec rekurencyjnie (jak `**/`)|
| `p.glob('**/*.py')` | równoważne `rglob` |

> 💡 `glob` i `rglob` zwracają generatory - pamieciowo efektywne
> przy duzych strukturach katalogów.

In [ ]:
from pathlib import Path

# Bezposrednie dzieci
for item in Path('.').iterdir():
    kind = 'DIR ' if item.is_dir() else 'FILE'
    print(f'{kind} {item.name}')

In [ ]:
from pathlib import Path

# Glob - szukanie plikow po wzorcu
py_files = list(Path('.').glob('*.ipynb'))
print(f'Notebooki w biezacym katalogu: {len(py_files)}')

# rglob - rekurencyjne przeszukiwanie
all_py = list(Path('.').rglob('*.py'))
print(f'Pliki .py rekurencyjnie: {len(all_py)}')

---

### 🐍 Ćwiczenia - przeszukiwanie katalogów

**Ćw. 7.** Napisz funkcję `list_files(directory, extension)` zwracającą
posortowaną listę nazw plików z podanym rozszerzeniem (np. `'.py'`)
bezpośrednio w `directory` (nie rekurencyjnie).

**Ćw. 8.** Napisz funkcję `dir_size(directory)` zwracającą łączny
rozmiar wszystkich plików w katalogu i podkatalogach (w bajtach).
Uzyj `rglob('*')` i `Path.stat().st_size`.

**Ćw. 9. *(Trudniejsze)*** Napisz funkcję `find_duplicates(directory)`
zwracającą słownik `{rozmiar: [lista sciezek]}` dla plików, które
mają ten sam rozmiar (potencjalne duplikaty). Uwzgledniaj tylko pliki
(nie katalogi).

In [ ]:
# Ćw. 7
from pathlib import Path

def list_files(directory: Path, extension: str) -> list[str]:
    ...


print(list_files(Path('.'), '.ipynb'))

In [ ]:
# Ćw. 8
from pathlib import Path

def dir_size(directory: Path) -> int:
    ...


print(f'{dir_size(Path(".")):.0f} bajtow')

In [ ]:
# Ćw. 9
from pathlib import Path
from collections import defaultdict

def find_duplicates(directory: Path) -> dict[int, list[Path]]:
    # hint: pogrupuj pliki po rozmiarze, zwroc tylko grupy z >= 2 plikami
    by_size: dict = defaultdict(list)
    for f in directory.rglob('*'):
        if f.is_file():
            ...
    return {size: paths for size, paths in by_size.items() if len(paths) >= 2}


duplicates = find_duplicates(Path('.'))
for size, paths in list(duplicates.items())[:3]:
    print(f'{size} B: {[p.name for p in paths]}')